# GPU Workload Advisor — Kaggle GPU Run

This notebook performs the one hardware-dependent acceptance test for the project. It builds the native CPU/OpenMP/CUDA benchmark, validates it with the Python test suite, runs one measured workload, asks the one-tool LLM agent to explain the evidence, and saves the resulting Markdown report.

Before running:

1. In Kaggle, open **Settings → Accelerator** and select **GPU**.
2. In the same settings pane, enable **Internet** for cloning and API access.
3. Put this project in a public GitHub repository.
4. Replace `REPOSITORY_URL` below.
5. For the final agent cell, add `OPENAI_API_KEY` under **Add-ons → Secrets** and enable it for this notebook. The native benchmark does not require the key.

> Run the cells in order. Start with matrix size 256, then use 1024 for your recorded demonstration after the small run succeeds.

In [ ]:
# 1. Confirm that Kaggle assigned an NVIDIA GPU and the CUDA compiler.
import shutil
import subprocess

for command in ("nvidia-smi", "nvcc", "cmake", "git"):
    if shutil.which(command) is None:
        raise RuntimeError(f"Required command is missing: {command}")

subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvcc", "--version"], check=True)

In [ ]:
# 2. Clone your project into Kaggle's writable working directory.
import os
from pathlib import Path

REPOSITORY_URL = "https://github.com/YOUR_USERNAME/gpu-workload-advisor.git"
PROJECT_DIR = Path("/kaggle/working/gpu-workload-advisor")

if "YOUR_USERNAME" in REPOSITORY_URL:
    raise ValueError("Replace REPOSITORY_URL with your public GitHub repository URL.")
if PROJECT_DIR.exists():
    raise FileExistsError(
        f"{PROJECT_DIR} already exists. Restart the Kaggle session before cloning again."
    )

subprocess.run(
    ["git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_DIR)],
    check=True,
)
os.chdir(PROJECT_DIR)
print(f"Project ready at {PROJECT_DIR}")

In [ ]:
# 3. Install the Python application and run GPU-independent tests.
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
    check=True,
)
subprocess.run([sys.executable, "-m", "pytest"], check=True)

In [ ]:
# 4. Compile the C++/OpenMP/CUDA executable.
BUILD_DIR = PROJECT_DIR / "build"

subprocess.run(
    [
        "cmake",
        "-S",
        str(PROJECT_DIR / "native"),
        "-B",
        str(BUILD_DIR),
        "-DCMAKE_BUILD_TYPE=Release",
    ],
    check=True,
)
subprocess.run(
    ["cmake", "--build", str(BUILD_DIR), "--parallel"],
    check=True,
)
print(f"Built {BUILD_DIR / 'benchmark_runner'}")

In [ ]:
# 5. Run one real benchmark and inspect its validated measurements.
import json
from app.benchmark import BenchmarkRunner
from app.config import Settings

MATRIX_SIZE = 256  # Change to 1024 after this small acceptance run succeeds.

run_settings = Settings(
    benchmark_executable=BUILD_DIR / "benchmark_runner",
    max_matrix_size=2048,
    benchmark_timeout_seconds=180,
    openai_model="gpt-5.4-mini",
)
runner = BenchmarkRunner(run_settings)
benchmark = runner.run(MATRIX_SIZE)
print(json.dumps(benchmark.model_dump(mode="json"), indent=2))

cuda_result = next(
    item for item in benchmark.measurements if item.implementation == "cuda"
)
if not cuda_result.available:
    raise RuntimeError(f"CUDA benchmark was unavailable: {cuda_result.error}")
if not all(item.correct is True for item in benchmark.measurements):
    raise RuntimeError("At least one correctness check failed; do not use these timings.")
print("All three implementations completed and passed correctness.")

## Optional LLM explanation

The next cell makes two small OpenAI API requests: one function-selection request and one evidence-grounded explanation request. It uses Kaggle Secrets so the API key is not written into the notebook. Skip this cell if you only want the native measurements.

In [ ]:
# 6. Load the API key securely and run the actual one-tool advisor.
from IPython.display import Markdown, display
from kaggle_secrets import UserSecretsClient
from app.agent import GpuAdvisor

os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")
advisor = GpuAdvisor(runner, run_settings)
advice = advisor.advise(
    f"Benchmark {MATRIX_SIZE} x {MATRIX_SIZE} matrix multiplication and explain the result."
)
display(Markdown(advice.report))

In [ ]:
# 7. Save the report as a downloadable Kaggle output.
REPORT_PATH = Path("/kaggle/working/gpu_workload_report.md")
REPORT_PATH.write_text(advice.report, encoding="utf-8")
print(f"Saved report to {REPORT_PATH}")

## What to record

For a portfolio result, save the GPU model, matrix size, all three latencies, correctness status, calculated CUDA-vs-CPU speedup, and whether transfers were included. Do not generalize beyond this hardware, implementation, matrix size, and run.